# Day 19 — Prompt Engineering Mastery

**Intern:** Sehar Andleeb · **Company:** Xeven Solutions · **Repo:** `ai-internship-xeven-2026`

**Stack:** Windows · UV · Python 3.13 · Groq free tier (`llama-3.3-70b-versatile`) · LangChain 1.x (LCEL) · `GROQ_API_KEY` in local `.env`.

Three tasks today: (1) compare zero-shot / few-shot / CoT on sentiment, (2) build a JSON prompt-template library, (3) advanced output control (JSON schema, markdown tables, code-gen) with validators.

## Research — Multi-Source Comparison



**Articles consulted (15 Jun 2026):** A = Prompt Engineering Guide / DAIR (promptingguide.ai) · B = IBM Think, *What is zero-shot prompting?* · C = DigitalOcean, *Prompt Engineering Best Practices*.

| Topic | ChatGPT  | Gemini  | Claude | A · DAIR | B · IBM | C · DigitalOcean |
|---|---|---|---|---|---|---|
| **Zero-shot** | Instruction only, no examples; fine for simple, common tasks | Relies on pretraining; cheapest, least steerable | Task description only; best baseline for easy tasks, weakest on multi-step reasoning | Strong zero-shot ability but falls short on complex tasks | Infers answer purely from pretraining; no demonstrations given | Lowest token cost; start here, escalate if it fails |
| **Few-shot** | 2–5 examples lift accuracy & consistency | Examples condition format/labels; pick diverse cases | 3–5 labeled examples sharply improve consistency and output format adherence | In-context demos steer the model; great for classification/format | Examples improve over zero-shot but can still miss on reasoning | Guides output format; keep examples representative |
| **Chain-of-Thought** | 'Let's think step by step' before answering | Best for math/logic/multi-step | Surface reasoning steps, then the label; helps ambiguous/edge cases, costs more tokens | Intermediate steps unlock complex reasoning; combine with few-shot | Decompose task into discrete steps; adds transparency | Break complex tasks into smaller steps |
| **System messages** | Set role, tone, constraints, format up front | Anchors persona; keeps behavior consistent | Define role + hard output rules at system level so they persist across inputs | (focus on techniques) | (focus on shots/CoT) | Specify format/structure to avoid drift |
| **Prompt patterns** | Persona, instruction, context, format, examples, constraints | Role + context + output structure framework | Task → role → format → context → constraints, layered together | Technique catalog (zero/few/CoT, auto-CoT) | Zero/few/CoT + tree-of-thought variants | Role/persona, chaining, reflection patterns |
| **Common pitfalls** | Vague asks, no format, conflicting rules | Assuming knowledge; overloaded prompts | Vagueness, missing output spec, conflicting instructions, unstated assumptions | Manual example crafting can be suboptimal | Zero/few-shot fail on hard reasoning — escalate | Too vague, multiple unrelated tasks, negative framing, skipping iteration |


## Theory (short)

- **Zero-shot** — describe the task, give no examples. Cheapest; best for simple, well-represented tasks.
- **Few-shot** — add 2–5 worked examples. Biggest practical win for classification and locking output format.
- **Chain-of-Thought** — ask for reasoning steps before the answer. Helps ambiguous/multi-step cases; costs more tokens and latency.
- **System messages** — set role, constraints, and output format once so they persist across every user input.
- **Prompt patterns** — persona · instruction · context · format · examples · constraints, composed together.
- **Pitfalls** — vagueness, no output spec, assumed knowledge, conflicting instructions. Fix by being specific and adding format + constraints.

In [1]:
# Dependencies ready.
# Install (run in your UV env, NOT in the notebook):
#   uv pip install langchain langchain-groq python-dotenv
print('Dependencies ready.')

Dependencies ready.


In [2]:
import os, sys
# Run task logic from the scripts/ folder so outputs are consistent.
if os.path.basename(os.getcwd()) != 'scripts':
    sys.path.insert(0, 'scripts')
    if os.path.isdir('scripts'):
        os.chdir('scripts')
import task1_technique_comparison as t1
import task2_template_library as t2
import task3_output_control as t3
print('Modules imported. LangChain available:', t1.LANGCHAIN_OK)

c:\Users\PMLS\Desktop\ai-internship-xeven-2026\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Modules imported. LangChain available: True


## Task 1 — Technique Comparison

**Live Groq run (llama-3.3-70b-versatile, 20 samples):**

| Technique | Accuracy | Avg latency | ~Tokens |
|---|---|---|---|
| Zero-shot | 90% | 1.0s | ~1445 |
| Few-shot | 100% | 1.29s | ~1445 |
| Chain-of-Thought | 95% | 2.80s | ~3002 |

Few-shot edged out CoT here: examples lock the label space cheaply, while CoT's reasoning roughly doubled tokens and latency for no gain on a task this simple. Cell below re-runs the pipeline (live if a `GROQ_API_KEY` is set, otherwise the offline mock).

In [3]:
key = os.getenv('GROQ_API_KEY')
live = bool(key) and t1.LANGCHAIN_OK
invoke = t1.make_live_invoker() if live else None
rows = [t1.run_technique(name, invoke, live)
        for name in ('zero_shot', 'few_shot', 'cot')]
for r in rows:
    print(f"{r['technique']:<11} acc={r['accuracy']:.1%} "
          f"lat={r['avg_latency_s']:.4f}s tok~{r['approx_total_tokens']} "
          f"mode={r['mode']}")

zero_shot   acc=90.0% lat=0.2943s tok~1445 mode=live
few_shot    acc=100.0% lat=1.3099s tok~1445 mode=live
cot         acc=95.0% lat=2.6654s tok~2897 mode=live


## Task 2 — Prompt Template Library

In [4]:
report = t2.render_all()
for item in report:
    print(item['task'], '->', 'OK' if item['valid'] else item['unfilled'])
print('All clean:', all(i['valid'] for i in report))

summarization -> OK
extraction -> OK
generation -> OK
analysis -> OK
All clean: True


## Task 3 — Advanced Output Control (validator robustness)

In [5]:
results = t3.robustness_suite()
passed = sum(c['passed'] for c in results)
for c in results:
    print(('PASS' if c['passed'] else 'FAIL'), c['case'], 'valid=' + str(c['valid']))
print(f'{passed}/{len(results)} validator cases behaved as expected.')

PASS json_clean valid=True
PASS json_fenced valid=True
PASS json_with_prose valid=True
PASS json_wrong_type valid=False
PASS json_missing_key valid=False
PASS table_good valid=True
PASS table_single_row valid=True
PASS table_ragged_row valid=False
PASS table_missing_separator valid=False
PASS code_documented valid=True
PASS code_no_docstring valid=False
11/11 validator cases behaved as expected.


## Summary
- Compared techniques live: **zero-shot 90% · few-shot 100% · CoT 95%**; few-shot best value, CoT ~2x cost for no gain on this task.
- Stored a 4-task prompt-template library as JSON with safe variable rendering; live summarization respected the 2-bullet / 25-word constraint.
- Wrote validators (JSON schema, markdown table, code-gen) and proved them on 11 good/edge/malformed cases; the live schema-bound JSON call returned with no schema errors.
- Day 19 complete and pushed to `ai-internship-xeven-2026`.